# Cloud Cover Analysis with Google Earth Engine

**Purpose:** Google Earth Engine (GEE) provides access to MODIS cloud observations at 1 km spatial resolution -- 25 times finer than the ERA5-based Open-Meteo approach. This higher resolution reveals spatial cloud patterns *within* flight boxes (e.g., persistent fog in valleys vs. clear ridgelines) and supports morning vs. afternoon cloud discrimination using Terra (~10:30 local) and Aqua (~13:30 local) overpasses. Use this notebook when you need detailed spatial cloud analysis or when morning/afternoon timing matters for your campaign.

| | |
|---|---|
| **Audience** | Intermediate to Advanced |
| **Runtime** | 5-15 minutes (GEE processing) |
| **Requires internet** | Yes |
| **Credentials required** | Google Earth Engine account (free for research) |
| **Optional dependencies** | `ee` (earthengine-api), `xarray` |
| **Uses example data** | Yes -- `exampledata/wdts.geojson` (Western US study areas) |

**What You Will Learn:**
- How to authenticate and retrieve cloud data from Google Earth Engine
- How MODIS Terra and Aqua enable morning vs. afternoon cloud analysis
- How to generate per-pixel spatial cloud fraction maps within flight boxes
- How to use GEE cloud data with the same downstream analysis functions as Open-Meteo

**Credential setup:** You need a Google Earth Engine account (free for research use). The first time you run this notebook, `ee.Authenticate()` will open a browser for OAuth login. After initial authentication, credentials are cached locally.

For the **Open-Meteo** (no-auth) workflow, visit simulation, forecasts, and climatology, see `cloud_analysis.ipynb`. Both notebooks produce the same output format, so all downstream functions (`simulate_visits`, plotting, etc.) work interchangeably.

We cover:
1. GEE authentication and setup
2. Retrieving cloud fraction via GEE (MODIS)
3. Morning vs afternoon cloud discrimination
4. Spatial cloud fraction maps

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import ee
ee.Authenticate()  # First time only
ee.Initialize()

from hyplan.clouds import (
    create_cloud_data_array_with_limit,
    fetch_cloud_fraction,
    fetch_cloud_fraction_spatial,
    plot_cloud_fraction_spatial,
    summarize_cloud_fraction_by_doy,
    plot_doy_cloud_fraction,
    simulate_visits,
    plot_yearly_cloud_fraction_heatmaps_with_visits,
)

In [ ]:
polygon_file = "exampledata/wdts.geojson"

gdf = gpd.read_file(polygon_file)
print(f"Study areas: {len(gdf)} polygons")
print(gdf[["Name"]].to_string(index=False))

## 1. Retrieving Cloud Fraction via GEE

**How it works:** The GEE path uses MODIS Terra/Aqua MOD09GA/MYD09GA surface reflectance with the `state_1km` quality assessment band to classify each pixel as clear, cloudy, or mixed. The cloud fraction for each polygon is the fraction of pixels classified as cloudy.

The output format is identical to the Open-Meteo path (a DataFrame with `polygon_id`, `year`, `day_of_year`, `cloud_fraction` columns), so all downstream functions (`simulate_visits`, `summarize_cloud_fraction_by_doy`, heatmap plotting, etc.) work interchangeably with either data source.

In [ ]:
# Via the low-level function
cloud_data_gee = create_cloud_data_array_with_limit(
    polygon_file=polygon_file,
    year_start=2003,
    year_stop=2022,
    day_start=1,
    day_stop=75,
)

# Or equivalently via the factory:
# cloud_data_gee = fetch_cloud_fraction(polygon_file, 2003, 2022, 1, 75, source="gee")

print(f"Retrieved {len(cloud_data_gee)} rows")
print(f"Columns: {list(cloud_data_gee.columns)}")
cloud_data_gee.head(10)

## 2. Morning vs Afternoon Cloud Discrimination

**Why this matters:** In many regions, cloud cover builds throughout the day. Coastal California, for example, often has clear mornings followed by afternoon marine layer intrusion. MODIS Terra (~10:30 local overpass) and Aqua (~13:30 local overpass) observe the same location at different times, providing a direct measurement of this diurnal cloud pattern.

If your campaign can be scheduled for morning-only flights, the effective number of clear days may be significantly higher than what daily-average cloud data suggests. The `satellite` and `split_satellite` parameters let you analyze this.

## 2. Morning vs Afternoon Cloud Discrimination

MODIS Terra (~10:30 local) and Aqua (~13:30 local) observe the same
location at different times of day. In many regions, cloud cover builds
throughout the day — so morning flights may have significantly better
conditions.

The `satellite` and `split_satellite` parameters let you analyze this.

In [ ]:
# Morning-only (Terra)
cloud_morning = create_cloud_data_array_with_limit(
    polygon_file, 2003, 2022, 1, 75, satellite="terra"
)

# Afternoon-only (Aqua)
cloud_afternoon = create_cloud_data_array_with_limit(
    polygon_file, 2003, 2022, 1, 75, satellite="aqua"
)

**Interpreting the comparison:** If the morning (Terra) climatology shows systematically lower cloud fractions than the afternoon (Aqua), it confirms a diurnal cloud buildup pattern. For campaign planning, this means scheduling flights for the morning could substantially increase the number of flyable days. The magnitude of the difference varies by site and season -- coastal sites typically show the strongest morning-afternoon contrast during summer months.

In [ ]:
## 3. Spatial Cloud Fraction Maps

**Why spatial detail matters:** Instead of a single cloud fraction per polygon, `fetch_cloud_fraction_spatial` produces a per-pixel raster showing *where* within each flight box is cloudier. This helps prioritize which sub-areas to fly first on marginal days -- if one end of a flight box is typically clearer, start there and work toward the cloudier end before conditions deteriorate.

In [ ]:
# Or keep both in one DataFrame with a 'satellite' column
cloud_split = create_cloud_data_array_with_limit(
    polygon_file, 2003, 2022, 1, 75, split_satellite=True
)
# cloud_split has columns: polygon_id, year, day_of_year, cloud_fraction, satellite
print(f"Columns: {list(cloud_split.columns)}")
print(f"Satellites: {sorted(cloud_split['satellite'].unique())}")
cloud_split.head(10)

## 3. Spatial Cloud Fraction Maps

Instead of a single cloud fraction per polygon, `fetch_cloud_fraction_spatial`
produces a per-pixel raster showing *where* within each flight box is
cloudier. This helps prioritize which sub-areas to fly first on marginal days.

**Interpreting the spatial maps:** The maps show the mean cloud fraction at each pixel, averaged over the 20-year record. Look for systematic spatial gradients -- for example, higher cloud fraction near the coast vs. inland, or in valleys vs. on ridges. These patterns are persistent and can inform flight line prioritization. If certain areas within a flight box are consistently cloudier, consider splitting the box into sub-regions with different scheduling priorities.

## Operational Takeaways

- **GEE provides 1 km resolution** cloud data vs. 25 km from Open-Meteo. Use GEE when spatial detail within flight boxes matters.
- **Morning vs. afternoon discrimination can significantly increase flyable days**, especially at coastal sites with diurnal cloud buildup. Always check whether scheduling morning-only flights is feasible.
- **Spatial cloud maps reveal persistent patterns** (coastal fog, valley clouds, orographic effects) that can inform flight line ordering on marginal days.
- **GEE requires authentication** but is free for research. Allow extra time for GEE queries (minutes vs. seconds for Open-Meteo).
- **Both sources produce identical output formats**, so you can use Open-Meteo for quick scoping and switch to GEE for detailed analysis without changing downstream code.

In [ ]:
# Returns dict of xarray DataArrays (one per polygon)
spatial = fetch_cloud_fraction_spatial(
    polygon_file,
    year_start=2003, year_stop=2022,
    day_start=1, day_stop=75,
    scale=1000,            # MODIS native resolution
    satellite="both",      # or "terra"/"aqua"
)

# spatial["bay_area"] is an xarray.DataArray with dims (latitude, longitude)
print(spatial["bay_area"])

In [ ]:
# Plot all polygons
fig = plot_cloud_fraction_spatial(spatial, polygon_file=polygon_file)